# PCA and SVD from Scratch

**Goal:** Implement PCA via eigendecomposition, PCA via SVD, incremental PCA, and Kernel PCA -- all in NumPy.

---

## Core Intuition

PCA finds the directions (principal components) along which the data has **maximum variance**. Projecting onto these directions gives the best low-dimensional representation (in the least-squares sense).

**Algorithm:**
1. Center data: $\tilde{X} = X - \bar{X}$
2. Compute covariance matrix: $C = \frac{1}{n-1} \tilde{X}^T \tilde{X}$
3. Eigendecompose: $C = V \Lambda V^T$
4. Sort eigenvectors by decreasing eigenvalue
5. Project: $Z = \tilde{X} V_k$ (keep top-k components)

**Key insight:** The $k$-th principal component maximizes variance subject to being orthogonal to the first $k-1$ components.

**Variance explained:** $\frac{\lambda_k}{\sum_j \lambda_j}$ for each component.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. PCA via Eigendecomposition

In [ ]:
class PCA_Eigen:
    """PCA via eigendecomposition of the covariance matrix."""
    
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.components_ = None      # principal axes (rows = components)
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None
        self.mean_ = None
    
    def fit(self, X):
        n_samples, n_features = X.shape
        if self.n_components is None:
            self.n_components = n_features
        
        # Step 1: Center
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # Step 2: Covariance matrix
        cov_matrix = (X_centered.T @ X_centered) / (n_samples - 1)
        
        # Step 3: Eigendecompose
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # Step 4: Sort by decreasing eigenvalue
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Keep top-k
        self.components_ = eigenvectors[:, :self.n_components].T  # (k, d)
        self.explained_variance_ = eigenvalues[:self.n_components]
        self.explained_variance_ratio_ = eigenvalues[:self.n_components] / np.sum(eigenvalues)
        
        return self
    
    def transform(self, X):
        X_centered = X - self.mean_
        return X_centered @ self.components_.T  # (n, k)
    
    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# Demo on 2D correlated data
np.random.seed(42)
theta = np.pi / 6
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta), np.cos(theta)]])
X_2d = np.random.randn(300, 2) @ np.diag([3, 0.5]) @ R.T + np.array([2, 1])

pca_eig = PCA_Eigen(n_components=2)
pca_eig.fit(X_2d)

print("Explained variance:", pca_eig.explained_variance_)
print("Explained variance ratio:", pca_eig.explained_variance_ratio_)
print("Principal components (rows):")
print(pca_eig.components_)

In [ ]:
# Visualize principal components on 2D data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Data with PC arrows
ax = axes[0]
ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.3, s=10, color='steelblue')
mean = pca_eig.mean_
for i, (comp, var) in enumerate(zip(pca_eig.components_, pca_eig.explained_variance_)):
    length = np.sqrt(var) * 2
    ax.annotate('', xy=mean + comp * length, xytext=mean,
                arrowprops=dict(arrowstyle='->', color=['red', 'green'][i], lw=3))
    ax.text(mean[0] + comp[0] * length * 1.1, mean[1] + comp[1] * length * 1.1,
            f'PC{i+1} ({pca_eig.explained_variance_ratio_[i]:.1%})',
            fontsize=12, fontweight='bold', color=['red', 'green'][i])

ax.set_title('Principal Components on 2D Data')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Right: Projected data
Z = pca_eig.transform(X_2d)
axes[1].scatter(Z[:, 0], Z[:, 1], alpha=0.3, s=10, color='steelblue')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('Data in Principal Component Space')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. PCA via SVD (More Numerically Stable)

Instead of computing the covariance matrix explicitly, we can use SVD directly on the centered data matrix:

$$\tilde{X} = U \Sigma V^T$$

The columns of $V$ are the principal components, and $\sigma_k^2 / (n-1)$ are the eigenvalues of the covariance matrix.

**Why SVD is better:**
- Avoids forming $X^TX$ which squares the condition number
- More numerically stable for ill-conditioned data
- Can use truncated SVD for efficiency

In [ ]:
class PCA_SVD:
    """PCA via Singular Value Decomposition (numerically stable)."""
    
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.components_ = None
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None
        self.mean_ = None
        self.singular_values_ = None
    
    def fit(self, X):
        n_samples, n_features = X.shape
        if self.n_components is None:
            self.n_components = min(n_samples, n_features)
        
        # Center
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # SVD
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        
        # Eigenvalues from singular values
        explained_variance = (S ** 2) / (n_samples - 1)
        total_var = np.sum(explained_variance)
        
        self.components_ = Vt[:self.n_components]  # (k, d)
        self.singular_values_ = S[:self.n_components]
        self.explained_variance_ = explained_variance[:self.n_components]
        self.explained_variance_ratio_ = explained_variance[:self.n_components] / total_var
        
        return self
    
    def transform(self, X):
        X_centered = X - self.mean_
        return X_centered @ self.components_.T
    
    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# Verify both implementations give same result
pca_svd = PCA_SVD(n_components=2)
pca_svd.fit(X_2d)

print("Eigen PCA variance:", pca_eig.explained_variance_)
print("SVD PCA variance:  ", pca_svd.explained_variance_)
print("\nDifference:", np.abs(pca_eig.explained_variance_ - pca_svd.explained_variance_))

## 3. Scree Plot and Variance Explained

Demo on a higher-dimensional dataset (Iris or synthetic).

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

pca_iris = PCA_SVD(n_components=4)
Z_iris = pca_iris.fit_transform(X_iris)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scree plot
axes[0].bar(range(1, 5), pca_iris.explained_variance_ratio_, color='steelblue', alpha=0.8)
axes[0].plot(range(1, 5), np.cumsum(pca_iris.explained_variance_ratio_), 'ro-', linewidth=2)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('Scree Plot (Iris Dataset)')
axes[0].set_xticks(range(1, 5))
axes[0].grid(True, alpha=0.3)
for i, ratio in enumerate(pca_iris.explained_variance_ratio_):
    axes[0].text(i + 1, ratio + 0.01, f'{ratio:.1%}', ha='center', fontweight='bold')

# 2D projection
colors = ['red', 'green', 'blue']
for i, name in enumerate(iris.target_names):
    mask = y_iris == i
    axes[1].scatter(Z_iris[mask, 0], Z_iris[mask, 1], c=colors[i], label=name, alpha=0.6, s=30)
axes[1].set_xlabel(f'PC1 ({pca_iris.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca_iris.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('Iris in 2D PCA Space')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Reconstruction error vs n_components
errors = []
for k in range(1, 5):
    pca_temp = PCA_SVD(n_components=k)
    Z_temp = pca_temp.fit_transform(X_iris)
    X_recon = pca_temp.inverse_transform(Z_temp)
    error = np.mean(np.sum((X_iris - X_recon) ** 2, axis=1))
    errors.append(error)

axes[2].plot(range(1, 5), errors, 'bo-', linewidth=2, markersize=8)
axes[2].set_xlabel('Number of Components')
axes[2].set_ylabel('Mean Reconstruction Error')
axes[2].set_title('Reconstruction Error vs Components')
axes[2].set_xticks(range(1, 5))
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Incremental (Online) PCA

When data is too large to fit in memory, process it in mini-batches. Uses the SVD update formula to incorporate new data without recomputing from scratch.

In [ ]:
class IncrementalPCA:
    """Incremental PCA that processes data in mini-batches.
    
    Uses the approach of updating the SVD with new data blocks.
    """
    
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.components_ = None
        self.mean_ = None
        self.var_ = None
        self.explained_variance_ = None
        self.n_samples_seen_ = 0
        self.singular_values_ = None
    
    def partial_fit(self, X):
        n_samples, n_features = X.shape
        
        if self.n_samples_seen_ == 0:
            # First batch: standard PCA
            self.mean_ = np.mean(X, axis=0)
            X_centered = X - self.mean_
            U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
            self.components_ = Vt[:self.n_components]
            self.singular_values_ = S[:self.n_components]
            self.n_samples_seen_ = n_samples
        else:
            # Incremental update
            n_old = self.n_samples_seen_
            n_new = n_samples
            n_total = n_old + n_new
            
            new_mean = np.mean(X, axis=0)
            
            # Updated mean
            updated_mean = (n_old * self.mean_ + n_new * new_mean) / n_total
            
            # Build matrix for SVD update
            X_centered = X - updated_mean
            mean_correction = np.sqrt(n_old * n_new / n_total) * (self.mean_ - new_mean)
            
            # Combine old singular values/vectors with new data
            old_S = np.diag(self.singular_values_)
            old_basis = self.components_  # (k, d)
            
            # Form combined matrix
            combined = np.vstack([
                old_S @ old_basis,       # weighted old components
                X_centered,              # new data
                mean_correction.reshape(1, -1)  # mean correction
            ])
            
            U, S, Vt = np.linalg.svd(combined, full_matrices=False)
            self.components_ = Vt[:self.n_components]
            self.singular_values_ = S[:self.n_components]
            self.mean_ = updated_mean
            self.n_samples_seen_ = n_total
        
        self.explained_variance_ = self.singular_values_ ** 2 / (self.n_samples_seen_ - 1)
        return self
    
    def transform(self, X):
        return (X - self.mean_) @ self.components_.T
    
    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_

# Test incremental PCA
ipca = IncrementalPCA(n_components=2)
batch_size = 30
for start in range(0, len(X_iris), batch_size):
    batch = X_iris[start:start + batch_size]
    ipca.partial_fit(batch)

Z_ipca = ipca.transform(X_iris)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, Z, title in [(axes[0], Z_iris[:, :2], 'Batch PCA'), (axes[1], Z_ipca, 'Incremental PCA')]:
    for i, name in enumerate(iris.target_names):
        mask = y_iris == i
        ax.scatter(Z[mask, 0], Z[mask, 1], c=colors[i], label=name, alpha=0.6, s=30)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Batch PCA vs Incremental PCA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Kernel PCA

Standard PCA only captures linear structure. Kernel PCA applies PCA in a high-dimensional feature space via the kernel trick:

1. Compute kernel matrix $K_{ij} = k(x_i, x_j)$
2. Center the kernel matrix: $\tilde{K} = K - 1_n K - K 1_n + 1_n K 1_n$ where $1_n = \frac{1}{n}\mathbf{11}^T$
3. Eigendecompose $\tilde{K} = U \Lambda U^T$
4. Components: $\alpha_k = \frac{1}{\sqrt{\lambda_k}} u_k$

In [ ]:
class KernelPCA:
    """Kernel PCA using RBF kernel."""
    
    def __init__(self, n_components=2, gamma=1.0):
        self.n_components = n_components
        self.gamma = gamma
    
    def _rbf_kernel(self, X1, X2):
        sq1 = np.sum(X1 ** 2, axis=1).reshape(-1, 1)
        sq2 = np.sum(X2 ** 2, axis=1).reshape(1, -1)
        dist_sq = sq1 + sq2 - 2 * X1 @ X2.T
        return np.exp(-self.gamma * dist_sq)
    
    def fit_transform(self, X):
        self.X_train = X.copy()
        n = X.shape[0]
        
        # Kernel matrix
        K = self._rbf_kernel(X, X)
        
        # Center kernel matrix
        one_n = np.ones((n, n)) / n
        K_centered = K - one_n @ K - K @ one_n + one_n @ K @ one_n
        self.K_train = K
        self.one_n = one_n
        
        # Eigendecompose
        eigenvalues, eigenvectors = np.linalg.eigh(K_centered)
        
        # Sort descending
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Keep top-k with positive eigenvalues
        self.eigenvalues_ = eigenvalues[:self.n_components]
        self.eigenvectors_ = eigenvectors[:, :self.n_components]
        
        # Normalize
        Z = self.eigenvectors_ * np.sqrt(np.maximum(self.eigenvalues_, 0))
        return Z
    
    def transform(self, X_new):
        K_new = self._rbf_kernel(X_new, self.X_train)
        n_train = self.X_train.shape[0]
        one_n_new = np.ones((X_new.shape[0], n_train)) / n_train
        K_centered = K_new - one_n_new @ self.K_train - K_new @ self.one_n + one_n_new @ self.K_train @ self.one_n
        
        Z = K_centered @ self.eigenvectors_ / np.sqrt(np.maximum(self.eigenvalues_, 1e-10))
        return Z

# Demo: circles data -- linear PCA vs Kernel PCA
from sklearn.datasets import make_circles
X_circ, y_circ = make_circles(n_samples=400, noise=0.05, factor=0.3, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original
axes[0].scatter(X_circ[y_circ == 0, 0], X_circ[y_circ == 0, 1], c='blue', alpha=0.5, s=15, label='Class 0')
axes[0].scatter(X_circ[y_circ == 1, 0], X_circ[y_circ == 1, 1], c='red', alpha=0.5, s=15, label='Class 1')
axes[0].set_title('Original Data')
axes[0].legend()
axes[0].set_aspect('equal')

# Linear PCA
pca_lin = PCA_SVD(n_components=2)
Z_lin = pca_lin.fit_transform(X_circ)
axes[1].scatter(Z_lin[y_circ == 0, 0], Z_lin[y_circ == 0, 1], c='blue', alpha=0.5, s=15)
axes[1].scatter(Z_lin[y_circ == 1, 0], Z_lin[y_circ == 1, 1], c='red', alpha=0.5, s=15)
axes[1].set_title('Linear PCA (classes overlap)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

# Kernel PCA
kpca = KernelPCA(n_components=2, gamma=15.0)
Z_kern = kpca.fit_transform(X_circ)
axes[2].scatter(Z_kern[y_circ == 0, 0], Z_kern[y_circ == 0, 1], c='blue', alpha=0.5, s=15)
axes[2].scatter(Z_kern[y_circ == 1, 0], Z_kern[y_circ == 1, 1], c='red', alpha=0.5, s=15)
axes[2].set_title('Kernel PCA (classes separated!)')
axes[2].set_xlabel('KPC1')
axes[2].set_ylabel('KPC2')

plt.suptitle('Linear PCA Fails on Non-Linear Data, Kernel PCA Succeeds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Image Compression with PCA

Each image can be treated as a high-dimensional vector. PCA finds the most important "eigen-images".

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_digits = digits.data  # (1797, 64)  -- 8x8 images
y_digits = digits.target

# Apply PCA and reconstruct with different numbers of components
fig, axes = plt.subplots(5, 8, figsize=(16, 10))

n_components_list = [1, 5, 10, 20, 64]
sample_indices = [0, 1, 2, 5, 10, 15, 20, 25]

for row, n_comp in enumerate(n_components_list):
    pca_dig = PCA_SVD(n_components=n_comp)
    Z = pca_dig.fit_transform(X_digits)
    X_recon = pca_dig.inverse_transform(Z)
    
    for col, idx in enumerate(sample_indices):
        axes[row, col].imshow(X_recon[idx].reshape(8, 8), cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(f'k={n_comp}', fontsize=12, rotation=0, labelpad=40)

plt.suptitle('Image Compression: Reconstruction with k Principal Components', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Compression ratio
for k in [5, 10, 20, 32]:
    original_size = 64  # features per image
    compressed_size = k   # components per image
    pca_temp = PCA_SVD(n_components=k)
    pca_temp.fit(X_digits)
    var_explained = np.sum(pca_temp.explained_variance_ratio_)
    print(f"k={k:3d}: compression ratio = {original_size/compressed_size:.1f}x, variance explained = {var_explained:.1%}")

## 7. Dimensionality Reduction: Digits in 2D

In [ ]:
pca_2d = PCA_SVD(n_components=2)
Z_2d = pca_2d.fit_transform(X_digits)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=y_digits, cmap='tab10', alpha=0.5, s=10)
plt.colorbar(scatter, label='Digit')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})')
plt.title('Digits Dataset Projected to 2D via PCA')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Compare with sklearn

In [ ]:
from sklearn.decomposition import PCA as SklearnPCA

sk_pca = SklearnPCA(n_components=4)
sk_pca.fit(X_iris)

our_pca = PCA_SVD(n_components=4)
our_pca.fit(X_iris)

print("Explained variance comparison:")
print(f"  Ours:    {our_pca.explained_variance_}")
print(f"  sklearn: {sk_pca.explained_variance_}")
print(f"  Max diff: {np.max(np.abs(our_pca.explained_variance_ - sk_pca.explained_variance_)):.2e}")

print("\nExplained variance ratio comparison:")
print(f"  Ours:    {our_pca.explained_variance_ratio_}")
print(f"  sklearn: {sk_pca.explained_variance_ratio_}")

---

## Interview Questions

### "PCA vs. Autoencoders?"
- PCA: linear, closed-form solution, finds orthogonal directions of max variance
- Autoencoders: non-linear (with non-linear activations), learned via gradient descent
- A linear autoencoder with MSE loss learns the same subspace as PCA
- Autoencoders can capture non-linear structure but are harder to train and interpret

### "When does PCA fail?"
- Non-linear relationships (use kernel PCA, autoencoders, t-SNE)
- When variance is not informative (high-variance noise direction)
- Discrete/categorical data
- When features have very different scales (need to standardize first)
- When the data manifold is not well approximated by a linear subspace

### "Whitening -- what and why?"
Whitening transforms data so that: (1) features are uncorrelated (PCA does this), and (2) each feature has unit variance (divide by $\sqrt{\lambda_k}$). Result: covariance = identity matrix. Used as preprocessing for ICA, some neural nets (batch norm is related).

### "PCA assumes linearity -- how to handle non-linear?"
- **Kernel PCA**: Apply PCA in a kernel-induced feature space (non-linear but fixed mapping)
- **Autoencoders**: Learn non-linear encoder/decoder
- **t-SNE / UMAP**: Non-linear visualization methods (not general-purpose dim reduction)
- **Manifold learning**: Isomap, LLE, etc.

### Notes
- **SVD connection:** PCA is just SVD of the centered data matrix. Truncated SVD = PCA without centering (useful for sparse matrices).
- **t-SNE/UMAP:** For visualization, t-SNE and UMAP often give better 2D/3D embeddings than PCA, but they don't preserve global structure as well and are not invertible.
- PCA is also used for **denoising** (reconstruct from top-k components, discarding noisy dimensions).

In [ ]:
print("Notebook 10 complete: PCA & SVD from Scratch")
print("="*50)
print("Implemented:")
print("  - PCA via eigendecomposition")
print("  - PCA via SVD")
print("  - Incremental (online) PCA")
print("  - Kernel PCA (RBF kernel)")
print("  - Image compression demo")
print("  - Validated against sklearn")